In [0]:
# olist_date_range_check.py
# Script: Chequeo de Intervalo de Fechas en la Capa Gold
# Ejecutar en Databricks (Python/PySpark)

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()

# *******************************************************************
# CONFIGURACIÓN DE UNITY CATALOG VOLUMES Y RUTAS
# *******************************************************************
CATALOG_NAME = "olist"
SCHEMA_NAME = "olist_csv"
GOLD_VOLUME_NAME = "gold_data"
gold_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{GOLD_VOLUME_NAME}/"

# *******************************************************************
# CARGA DE LA TABLA GOLD Y CÁLCULO DEL INTERVALO
# *******************************************************************
try:
    # 1. Cargar las features de la capa Gold
    features_spark = spark.read.parquet(gold_path + "customer_features")
    print(f"Features cargadas desde: {gold_path}customer_features")

    # 2. Calcular la fecha mínima y máxima de los registros
    # Se usan las columnas 'first_purchase_ts' y 'last_purchase_ts'
    date_range = features_spark.select(
        F.min("first_purchase_ts").alias("fecha_inicio"),
        F.max("last_purchase_ts").alias("fecha_fin")
    ).collect()[0]

    fecha_inicio = date_range["fecha_inicio"]
    fecha_fin = date_range["fecha_fin"]
    
    print("\n--- INTERVALO DE TIEMPO DE LOS REGISTROS DE CLIENTES ---")
    if fecha_inicio and fecha_fin:
        print(f"Fecha de la primera compra registrada (inicio del dataset): {fecha_inicio}")
        print(f"Fecha de la última compra registrada (fin del dataset): {fecha_fin}")
    else:
        print("Advertencia: No se encontraron columnas de fecha válidas ('first_purchase_ts' o 'last_purchase_ts') para calcular el intervalo.")
    print("------------------------------------------------------\n")
    
except Exception as e:
    print(f"\n¡ERROR CRÍTICO!")
    print(f"No se pudo cargar la tabla de features o calcular el intervalo de fechas. Verifique:")
    print(f"  - Que la ruta de la tabla Gold sea correcta: {gold_path}customer_features")
    print(f"  - Que el clúster esté corriendo y tenga permisos.")
    print(f"Detalles del error: {e}")